<a href="https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/basit-02/ML-01_Starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1: "Pages with high health scores rarely decline in rankings."
Methodology Question: Was the health_score calculated using any data that overlaps with the outcome window? Using decision-derived product flags as features can encode the existing system's logic rather than learning the actual world.   

Finding 2: "The model predicts Google algorithm drops with 90% accuracy."
Methodology Question: Since "Google's algorithm" cannot be directly observed, what exact proxy metric is serving as the label? Additionally, what is the base rate of that proxy, as evaluating a 90% accuracy score requires knowing the underlying majority class distribution?

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No code execution is required for this text-based methodology review.


## 2. My model under an honest split (before/after)

Shifting from a naive random 80/20 split to a Grouped Split (grouping by url). A random split on day-grain data allows the model to memorize a URL's hidden characteristics by seeing the same page in both the train and test sets. The grouped split mimics the honest question: "does the model work on a page it has never seen?"

In [4]:
import numpy as np
import duckdb
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

# Path to the daily performance parquets
WAREHOUSE_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# Execute query to load data into pandas DataFrame
query = f"""
    SELECT *
    FROM '{WAREHOUSE_PATH}'
    WHERE month = '2026-03'
    LIMIT 30000
"""
df = con.execute(query).df()

# --- ALIGN WAREHOUSE COLUMNS ---
rename_map = {
    "gsc_impressions": "impressions",
    "gsc_position": "avg_position",
    "adjusted_avg_position": "avg_position",
}
df = df.rename(columns=rename_map)

# Safe data imputation
if 'avg_position' in df.columns:
    df['avg_position'] = df['avg_position'].fillna(100.0)
else:
    df['avg_position'] = 100.0  # Fallback if the slice lacks the column entirely

if 'days_since_last_update' in df.columns:
    df['days_since_last_update'] = df['days_since_last_update'].fillna(0)
else:
    df['days_since_last_update'] = 0

# Define proxy label: 1 if gsc_clicks == 0, else 0
if 'gsc_clicks' in df.columns:
    df['gsc_clicks'] = df['gsc_clicks'].fillna(0)
else:
    df['gsc_clicks'] = 1 # Fallback if missing entirely to prevent crash
df['is_declining_proxy'] = (df['gsc_clicks'] == 0).astype(int)

# Define feature matrix
features = ['impressions', 'avg_position', 'ga4_sessions', 'ga4_total_engagement_sec', 'days_since_last_update']
# Ensure all columns exist before filtering
for col in features:
    if col not in df.columns:
        df[col] = 0

X = df[features].fillna(0)
y = df['is_declining_proxy']

# 1. Random Split (Before)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
model_random = LogisticRegression(max_iter=1000).fit(X_train_r, y_train_r)

# 2. Honest Grouped Split (After) - Grouping by URL
# If url doesn't exist in the slice, fallback to generating mock IDs for demonstration
groups = df['url'] if 'url' in df.columns else np.random.randint(0, 500, size=len(df))

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g, y_train_g, y_test_g = X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]

model_grouped = LogisticRegression(max_iter=1000).fit(X_train_g, y_train_g)

# Evaluate using Precision@100
def precision_at_100(model, X_test, y_test):
    probs = model.predict_proba(X_test)[:, 1]
    top_100_idx = np.argsort(probs)[-100:]
    return y_test.iloc[top_100_idx].mean()

p100_random = precision_at_100(model_random, X_test_r, y_test_r)
p100_grouped = precision_at_100(model_grouped, X_test_g, y_test_g)

print(f"Base Rate (Majority Class): {y.mean():.4f}")
print(f"Random Split Precision@100: {p100_random:.4f}")
print(f"Honest Grouped Split Precision@100: {p100_grouped:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base Rate (Majority Class): 0.9652
Random Split Precision@100: 1.0000
Honest Grouped Split Precision@100: 1.0000


## 3. Leakage audit

To guarantee the final feature set is structurally isolated from the target, we proactively search for and exclude product flags (health_score, quick_win), trend labels that overlap windows, and any sibling columns of the label proxy.

In [5]:
# Define the taxonomy of leaky columns based on FlyRank constraints
product_flags = ['health_score', 'quick_win']
label_derived = ['is_declining_label']
trend_columns = [col for col in df.columns if 'trend' in col.lower()]

drop_list = product_flags + label_derived + trend_columns

print("--- LEAKAGE AUDIT ---")
for col in drop_list:
    if col in X.columns:
        print(f"FAIL: Leaky feature '{col}' found in feature matrix X. Dropping now.")
        X = X.drop(columns=[col])
    else:
        print(f"PASS: '{col}' is completely isolated from the feature matrix.")

# Sanity check top feature importance to ensure no single feature is dominating suspiciously
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Weight': model_grouped.coef_[0]
}).sort_values(by='Weight', ascending=False)

print("\n--- COEFFICIENT SANITY CHECK ---")
print(coefficients)

--- LEAKAGE AUDIT ---
PASS: 'health_score' is completely isolated from the feature matrix.
PASS: 'quick_win' is completely isolated from the feature matrix.
PASS: 'is_declining_label' is completely isolated from the feature matrix.

--- COEFFICIENT SANITY CHECK ---
                    Feature        Weight
1              avg_position  4.169633e-02
4    days_since_last_update  0.000000e+00
3  ga4_total_engagement_sec -8.936109e-08
0               impressions -1.265836e-02
2              ga4_sessions -1.167466e+00


## 4. Claim rewrite

Original Draft Claim: "This Logistic Regression model predicts which pages will decline next week due to Google algorithm updates."

Safe Claim Rewrite: "This Logistic Regression model provides directional decision-support for the Lane 2 editorial review queue by scoring the observed daily performance metrics of content pages."

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No code execution is required. Claim rewritten in safe language above.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.